## ResNet

In [ ]:
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import f1_score
import torch
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.45, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder('../data/combined_split/train', transform=transform)

# Get subset of training data
generator = torch.Generator().manual_seed(10)
num_samples = int(len(train_dataset) * 0.05)
train_subset, _ = random_split(train_dataset, [num_samples, len(train_dataset) - num_samples], generator)

val_dataset = ImageFolder('../data/combined_split/val', transform=transform)
test_dataset = ImageFolder('../data/combined_split/test', transform=transform)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)    # load subset
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    # Determine whether to use GPU (if available) or CPU
    device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")


    for epoch in range(num_epochs):
        # Set the model to training mode
        model.train()

        # Initialize running loss and correct predictions count for training
        running_loss = 0.0
        running_corrects = 0

        # Iterate over the training data loader
        for inputs, labels in train_loader:
            # Move inputs and labels to the device (GPU or CPU)
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Reset the gradients to zero before the backward pass
            optimizer.zero_grad()

            # Forward pass: compute the model output
            outputs = model(inputs)
            # Get the predicted class (with the highest score)
            _, preds = torch.max(outputs, 1)
            # Compute the loss between the predictions and actual labels
            loss = criterion(outputs, labels)

            # Backward pass: compute gradients
            loss.backward()
            # Perform the optimization step to update model parameters
            optimizer.step()

            # Accumulate the running loss and the number of correct predictions
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        # Compute average training loss and accuracy for this epoch
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = running_corrects.float() / len(train_loader.dataset)

        # Set the model to evaluation mode for validation
        model.eval()
        # Initialize running loss and correct predictions count for validation
        running_loss = 0.0
        running_corrects = 0
        all_preds = []
        all_labels = []

        # Disable gradient computation for validation (saves memory and computations)
        with torch.no_grad():
            # Iterate over the validation data loader
            for inputs, labels in val_loader:
                # Move inputs and labels to the device (GPU or CPU)
                inputs = inputs.to(device)
                labels = labels.to(device)

                # Forward pass: compute the model output
                outputs = model(inputs)
                # Get the predicted class (with the highest score)
                _, preds = torch.max(outputs, 1)
                # Compute the loss between the predictions and actual labels
                loss = criterion(outputs, labels)

                # Accumulate the running loss and the number of correct predictions
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

                # Accumulate predictions and labels for F1
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # Compute average validation loss and accuracy for this epoch
        val_loss = running_loss / len(val_loader.dataset)
        val_acc = running_corrects.float() / len(val_loader.dataset)
        val_f1 = f1_score(all_labels, all_preds, average='macro')

        # 2. Log Metrics to WandB
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1
        })

        # print(f'Epoch [{epoch+1}/{num_epochs}], train loss: {train_loss:.4f}, train acc: {train_acc:.4f}, val loss: {val_loss:.4f}, val acc: {val_acc:.4f}, val f1: {val_f1:.4f}')


from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate_model(model, test_loader, device):
    # Initialize dictionaries to store correct and total predictions
    correct_pred = {classname: 0 for classname in test_loader.dataset.classes}
    total_pred = {classname: 0 for classname in test_loader.dataset.classes}

    # Set the model to evaluation mode
    model.eval()

    # Track the ground truth labels and predictions
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            # Move the inputs and labels to the device
            inputs = inputs.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            # Collect predictions and labels for metric calculations
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

            # Update the correct and total predictions
            for label, prediction in zip(labels, preds):
                classname = test_loader.dataset.classes[label]
                if label == prediction:
                    correct_pred[classname] += 1
                total_pred[classname] += 1

    # Calculate accuracy per class
    accuracy_per_class = {classname: correct_pred[classname] / total_pred[classname] if total_pred[classname] > 0 else 0
                          for classname in test_loader.dataset.classes}

    # Calculate overall accuracy
    overall_accuracy = accuracy_score(all_labels, all_preds)

    f1 = f1_score(all_labels, all_preds, average='macro')

    # Print the evaluation results
    print("Accuracy per class:")
    for classname, accuracy in accuracy_per_class.items():
        print(f"{classname}: {accuracy:.4f}")

    print()
    print(f"Overall Accuracy: {overall_accuracy:.4f}, F1 Score: {f1:.4f}")
    return f1

In [ ]:
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import os
from PIL import Image
from sklearn.metrics import f1_score
from tqdm import tqdm
import wandb
import gc

def train_percent(percent, epoch_num):
    f1_scores = []
    for i in range(10):
        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.45, 0.406], std=[0.229, 0.224, 0.225])
        ])

        train_dataset = ImageFolder('../data/combined_split/train', transform=transform)

        # Get subset of training data
        num_samples = int(len(train_dataset) * percent)
        train_subset, _ = random_split(train_dataset, [num_samples, len(train_dataset) - num_samples])

        val_dataset = ImageFolder('../data/combined_split/val', transform=transform)
        test_dataset = ImageFolder('../data/combined_split/test', transform=transform)

        train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)    # load subset
        val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

        # RESNET -----------------------
        resnet = models.resnet50(pretrained=True)
        num_classes = len(train_dataset.classes)
        resnet.fc = torch.nn.Linear(resnet.fc.in_features, num_classes)

        # Define loss function and optimizer
        criterion = torch.nn.CrossEntropyLoss()
        optimizer = torch.optim.SGD(resnet.fc.parameters(), lr=0.001, momentum=0.9)

        # # 1. Login and Initialize
        # wandb.login()

        run = wandb.init(
            entity="rchan192-university-of-california-riverside",
            project="icl_hlbdetection",  
            config={
                "learning_rate": 0.001,
                "architecture": "resnet",
                "dataset": "combined",
                "epochs": epoch_num,
                "batch_size": 32,
                "percent": percent
            }
        )


        # Run training
        model = resnet.to(device)
        train(model, train_loader, val_loader, criterion, optimizer, num_epochs=epoch_num)
        f1_scores.append(evaluate_model(model, test_loader, device))

        # 3. Close the WandB run
        run.finish()

        # Clean up memory
        del model, optimizer, train_loader
        torch.cuda.empty_cache()
        gc.collect()

    return f1_scores

percents = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]
epochs = [28, 12, 15, 24, 33, 10, 16, 18, 19, 8, 15]

results = []
for num, epoch_num in zip(percents, epochs):
    f1_scores = train_percent(num, epoch_num)
    results.append({"Percent": num, "F1_Scores": f1_scores})
    
df = pd.DataFrame(results)
df.to_csv('resnetTrials.csv', index=False)



## VGG-19

In [ ]:
import numpy as np
import pandas as pd
import torch
import torchvision.models as models
from transformers import CLIPProcessor, CLIPModel, AutoModel, AutoImageProcessor
from tqdm import tqdm
import wandb

percents = [0.90, 1.00]

for percent in percents:

    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.45, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dataset = ImageFolder('../data/combined_split/train', transform=transform)

    # Get subset of training data
    generator = torch.Generator().manual_seed(10)
    num_samples = int(len(train_dataset) * percent)
    train_subset, _ = random_split(train_dataset, [num_samples, len(train_dataset) - num_samples], generator)

    val_dataset = ImageFolder('../data/combined_split/val', transform=transform)
    test_dataset = ImageFolder('../data/combined_split/test', transform=transform)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)    # load subset
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    # VGG19 -----------------------
    # Model setup
    vgg19 = models.vgg19(pretrained=True)
    num_classes = len(train_dataset.classes)
    vgg19.classifier[6] = torch.nn.Linear(4096, num_classes)
    model = vgg19.to(device)

    # Define loss function and optimizer
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

    # 1. Login and Initialize
    wandb.login()

    run = wandb.init(
        entity="rchan192-university-of-california-riverside",
        project="icl_hlbdetection",
        config={
            "learning_rate": 0.001,
            "architecture": "vgg19",
            "dataset": "CitrusUAT",
            "epochs": 20,
            "batch_size": 32
        }
    )

    # Run training
    model = vgg19.to(device)
    train(model, train_loader, val_loader, criterion, optimizer, num_epochs=20)

    # 3. Close the WandB run
    run.finish()

    evaluate_model(model, test_loader, device)

## CLIP

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import random
import os
from PIL import Image
from sklearn.metrics import f1_score
from transformers import CLIPProcessor, CLIPModel

# load model
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# split 
train_images = []
train_labels = []
train_data_path = os.path.join("..", "data", "combined_split", "train")
for root, dirs, files in os.walk(train_data_path):
    if len(dirs) != 0:
        labels = dirs
    else:
        label = root.replace(train_data_path, "").replace("_", " ")
        label = label.replace("HLB", "Huanglongbing")
        train_images.extend([os.path.join(root, file) for file in files])
        train_labels.extend([label] * len(files))

val_images = []
val_labels = []
val_data_path = os.path.join("..", "data", "combined_split", "val")
for root, dirs, files in os.walk(val_data_path):
    if len(dirs) != 0:
        labels = dirs
    else:
        label = root.replace(val_data_path, "").replace("_", " ")
        label = label.replace("HLB", "Huanglongbing")
        val_images.extend([os.path.join(root, file) for file in files])
        val_labels.extend([label] * len(files))

test_images = []
test_labels = []
test_data_path = os.path.join("..", "data", "combined_split", "test")
for root, dirs, files in os.walk(test_data_path):
    if len(dirs) != 0:
        labels = dirs
    else:
        label = root.replace(test_data_path, "").replace("_", " ")
        label = label.replace("HLB", "Huanglongbing")
        test_images.extend([os.path.join(root, file) for file in files])
        test_labels.extend([label] * len(files))

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
clip_model = clip_model.to(device)
clip_model.float()

# create dataset
class clipdataset():
    def __init__(self, image_paths, labels):
        self.image_paths = image_paths
        self.labels = labels
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]

        return image, label

    def getsubset(self, frac):
        num_samples = int(len(self.image_paths) * frac)
        generator = torch.Generator().manual_seed(10)
        subset, _ = random_split(self, [num_samples, len(self.image_paths) - num_samples], generator)
        return subset

def collate_fn(batch):
    images, texts = zip(*batch)
    inputs = clip_processor(
        text=list(texts),
        images=list(images),
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    inputs["raw_texts"] = list(texts)
    return inputs

clip_train_dataloader = DataLoader(clipdataset(train_images, train_labels).getsubset(0.05), batch_size=32, shuffle=True, collate_fn=collate_fn)

clip_val_dataloader = DataLoader(clipdataset(val_images, val_labels), batch_size=64, shuffle=False, collate_fn=collate_fn)

clip_test_dataloader = DataLoader(clipdataset(test_images, test_labels), batch_size=64, shuffle=False, collate_fn=collate_fn)

# loss function + num of correct guesses
def clip_criterion(image_embeds, text_embeds, logit_scale):
    # normalize
    image_embeds = F.normalize(image_embeds, p=2, dim=-1)
    text_embeds = F.normalize(text_embeds, p=2, dim=-1)

    logits_per_image = logit_scale * (image_embeds @ text_embeds.T)
    logits_per_text = logits_per_image.T

    batch_size = image_embeds.size(0)
    labels = torch.arange(batch_size, device=image_embeds.device)

    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)

    return (loss_i + loss_t) / 2

lr = 1e-5
optimizer = optim.AdamW(clip_model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-6, weight_decay=0.01)

class_prompts = sorted(list(set(train_labels)))
class_to_idx = {t: i for i, t in enumerate(class_prompts)}

with torch.no_grad():
    text_inputs = clip_processor(
        text=class_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    text_inputs = {k: v.to(device) for k, v in text_inputs.items()}

    text_feats = clip_model.get_text_features(**text_inputs)
    if not torch.is_tensor(text_feats):
        text_feats = text_feats.pooler_output

    class_text_embeds = F.normalize(text_feats, dim=-1)

# train
def clip_train(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    # recompute embeddings
    with torch.no_grad():
        text_feats = clip_model.get_text_features(**text_inputs)
        if not torch.is_tensor(text_feats):
            text_feats = text_feats.pooler_output

        class_text_embeds = F.normalize(text_feats, dim=-1)
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        
        all_preds = []
        all_targets = []

        for batch in train_loader:
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_loss=False
            )

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            logit_scale = model.logit_scale.exp().clamp(1, 100)
            loss = criterion(image_embeds, text_embeds, logit_scale)

            class_logits = F.normalize(image_embeds, dim=-1) @ class_text_embeds.T
            preds = class_logits.argmax(dim=-1)
            targets = torch.tensor([class_to_idx[t] for t in batch["raw_texts"]], device=device)
            correct = (preds == targets).float()

            total_loss += loss.item() * len(pixel_values)
            total_correct += correct.sum().item()

            all_preds.append(preds.detach().cpu())
            all_targets.append(targets.detach().cpu())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        avg_loss = total_loss / len(train_loader.dataset)
        avg_acc = total_correct / len(train_loader.dataset)

        all_preds = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()
        f1 = f1_score(all_targets, all_preds, average='macro')

        # validation set
        model.eval()
        total_loss_v = 0
        total_correct_v = 0

        all_preds_v = []
        all_targets_v = []
        
        with torch.no_grad():
            for batch in val_loader:
                pixel_values = batch['pixel_values'].to(device)
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
            
                outputs = model(
                    pixel_values=pixel_values,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    return_loss=False
                )

                image_embeds = outputs.image_embeds
                text_embeds = outputs.text_embeds

                logit_scale = model.logit_scale.exp().clamp(1, 100)
                loss = criterion(image_embeds, text_embeds, logit_scale)

                class_logits = F.normalize(image_embeds, dim=-1) @ class_text_embeds.T
                preds = class_logits.argmax(dim=-1)
                targets = torch.tensor([class_to_idx[t] for t in batch["raw_texts"]], device=device)
                correct = (preds == targets).float()

                total_loss_v += loss.item() * len(pixel_values)
                total_correct_v += correct.sum().item()
                
                all_preds_v.append(preds.detach().cpu())
                all_targets_v.append(targets.detach().cpu())
        avg_loss_v = total_loss_v / len(val_loader.dataset)
        avg_acc_v = total_correct_v / len(val_loader.dataset)

        all_preds_v = torch.cat(all_preds_v).numpy()
        all_targets_v = torch.cat(all_targets_v).numpy()
        f1_v = f1_score(all_targets_v, all_preds_v, average='macro')
        
        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {avg_loss:.4f}, Training Accuracy: {avg_acc:.4f}, Training F1 Score: {f1:.4f}, Validation Loss: {avg_loss_v:.4f}, Validation Accuracy: {avg_acc_v:.4f}, Validation F1 Score: {f1_v:.4f}")
        run.log({"train loss": avg_loss,"train acc": avg_acc, "train f1 score": f1, "val loss": avg_loss_v, "val acc": avg_acc_v, "val f1 score": f1_v})

def clip_evaluate(model, test_loader, criterion, device):
    # recompute embeddings
    with torch.no_grad():
        text_feats = clip_model.get_text_features(**text_inputs)
        if not torch.is_tensor(text_feats):
            text_feats = text_feats.pooler_output

        class_text_embeds = F.normalize(text_feats, dim=-1)

    model.eval()
    total_loss = 0
    total_correct = 0

    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in test_loader:
            pixel_values = batch['pixel_values'].to(device)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
        
            outputs = model(
                pixel_values=pixel_values,
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_loss=False
            )

            image_embeds = outputs.image_embeds
            text_embeds = outputs.text_embeds

            logit_scale = model.logit_scale.exp().clamp(1, 100)
            loss = criterion(image_embeds, text_embeds, logit_scale)

            class_logits = F.normalize(image_embeds, dim=-1) @ class_text_embeds.T
            preds = class_logits.argmax(dim=-1)
            targets = torch.tensor([class_to_idx[t] for t in batch["raw_texts"]], device=device)
            correct = (preds == targets).float()

            total_loss += loss.item() * len(pixel_values)
            total_correct += correct.sum().item()

            all_preds.append(preds.detach().cpu())
            all_targets.append(targets.detach().cpu())
    avg_loss = total_loss / len(test_loader.dataset)
    avg_acc = total_correct / len(test_loader.dataset)

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    f1 = f1_score(all_targets, all_preds, average='macro')
    
    print(f"Final Average Loss: {avg_loss:.4f}, Final Average Accuracy: {avg_acc:.4f}, Final F1 Score: {f1:.4f}")
    run.log({"final acc": avg_acc, "final f1": f1})

num_epochs = 3
import wandb

run = wandb.init(
    entity="rchan192-university-of-california-riverside",
    project="my-awesome-project",
    config={
        "learning_rate": lr,
        "architecture": "CLIP",
        "dataset": "CitrusUAT + Orange Leaves for HLB",
        "epochs": num_epochs,
    },
)

clip_train(clip_model, clip_train_dataloader, clip_val_dataloader, clip_criterion, optimizer, num_epochs, device)
clip_evaluate(clip_model, clip_test_dataloader, clip_criterion, device)

run.finish()

## DINOV2

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import random_split
from transformers import AutoModel, AutoImageProcessor, Trainer, TrainingArguments, TrainerCallback
from torchvision import datasets
import os
from sklearn.metrics import f1_score

if torch.cuda.is_available():
    device = torch.device("cuda")
# elif torch.backends.mps.is_available():
#     device = torch.device("mps")
else:
    device = torch.device("cpu")

# load model
# pretrained_model_name = "facebook/dinov2-base"
# dinov2_processor = AutoImageProcessor.from_pretrained(pretrained_model_name)
# dinov2_backbone = AutoModel.from_pretrained(pretrained_model_name).to(device)

# split 
train_data_path = os.path.join("..", "data", "combined_split", "train")
val_data_path = os.path.join("..", "data", "combined_split", "val")
test_data_path = os.path.join("..", "data", "combined_split", "test")

# dataset wrapper
class dinov2dataset():
    def __init__(self, root, processor):
        self.ds = datasets.ImageFolder(root)
        self.processor = processor
    
    def __len__(self):
        return len(self.ds)
    
    def __getitem__(self, idx):
        img, label = self.ds[idx]
        inputs = self.processor(images=img, return_tensors="pt")
        return {"pixel_values": inputs["pixel_values"].squeeze(0), "labels": torch.tensor(label)}
    
    def getsubset(self, frac, generator=None):
        num_samples = int(len(self.ds) * frac)
        # generator = torch.Generator().manual_seed(7)
        if generator == None:
            subset, _ = random_split(self, [num_samples, len(self.ds) - num_samples])
        else:
            subset, _ = random_split(self, [num_samples, len(self.ds) - num_samples], generator)
        return subset

# frac = 0.9
# train_ds = dinov2dataset(train_data_path, dinov2_processor).getsubset(frac)
# val_ds = dinov2dataset(val_data_path, dinov2_processor)
# test_ds = dinov2dataset(test_data_path, dinov2_processor)

num_classes = len(os.listdir(train_data_path))

# custom -> wrap backbone and head
class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(backbone.config.hidden_size, num_classes)

    def forward(self, pixel_values, labels=None):
        out = self.backbone(pixel_values=pixel_values)
        logits = self.classifier(out.last_hidden_state[:, 0])
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}
    
# dinov2 = DINOv2Classifier(dinov2_backbone, num_classes)

# train
# lr = 1e-5
# num_epochs = 6
# training_args = TrainingArguments(
#     output_dir="./results",
#     eval_strategy="epoch",
#     logging_strategy="epoch",
#     per_device_train_batch_size=32,
#     per_device_eval_batch_size=64,
#     num_train_epochs=num_epochs,
#     learning_rate=lr,
#     save_steps=500,
#     save_total_limit=2,
# )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    f1 = f1_score(labels, preds, average="macro")
    acc = (preds == labels).mean()
    return {"f1": f1, "acc": acc}

# print per epoch while training
class EpochMetricsCallback(TrainerCallback):
    def __init__(self):
        self.metrics = {}

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        self.metrics.update(metrics)

        # if "eval_train_f1" in self.metrics and "eval_val_f1" in self.metrics:
        #     epoch = int(state.epoch)
        #     print(f"Epoch {epoch}/{num_epochs}, Training Loss: {self.metrics.get('eval_train_loss', 0):.4f}, Training Accuracy: {self.metrics.get('eval_train_acc', 0):.4f}, Training F1 Score: {self.metrics.get('eval_train_f1', 0):.4f}, Validation Loss: {self.metrics.get('eval_val_loss', 0):.4f}, Validation Accuracy: {self.metrics.get('eval_val_acc', 0):.4f}, Validation F1 Score: {self.metrics.get('eval_val_f1', 0):.4f}")
        #     # run.log({"train loss": self.metrics.get('eval_train_loss', 0),"train acc": self.metrics.get('eval_train_acc', 0), "train f1 score": self.metrics.get('eval_train_f1', 0), "val loss": self.metrics.get('eval_val_loss', 0), "val acc": self.metrics.get('eval_val_acc', 0), "val f1 score": self.metrics.get('eval_val_f1', 0)})
        #     self.metrics = {}

# dinov2_trainer = Trainer(
#     model=dinov2,
#     args=training_args,
#     train_dataset=train_ds,
#     eval_dataset={"train": train_ds, "val": val_ds},
#     compute_metrics=compute_metrics,
#     callbacks=[EpochMetricsCallback()],
# )

# evaluate on test set
def dinov2_evaluate(trainer, dataset):
    results = trainer.predict(dataset)
    preds = results.predictions.argmax(axis=-1)
    labels = results.label_ids
    f1 = f1_score(labels, preds, average="macro")
    acc = (preds == labels).mean()

    print(f"Final Average Accuracy: {acc:.4f}, Final F1 Score: {f1:.4f}")
    # run.log({"final acc": acc, "final f1": f1})

    return f1

# import wandb
# run = wandb.init(
#     entity="rchan192-university-of-california-riverside",
#     project="my-awesome-project",
#     config={
#         "learning_rate": lr,
#         "architecture": "DINOv2",
#         "dataset": f"{frac * 100}% CitrusUAT + Orange Leaves for HLB",
#         "epochs": num_epochs,
#     },
# )

# dinov2_trainer.train()
# dinov2_evaluate(dinov2_trainer, test_ds)

# run.finish()

## DINOV3

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoImageProcessor, Trainer, TrainingArguments, TrainerCallback
from torchvision import datasets
import os
from sklearn.metrics import f1_score

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# load model
pretrained_model_name = "facebook/dinov3-vitb16-pretrain-lvd1689m"
dinov3_processor = AutoImageProcessor.from_pretrained(pretrained_model_name)
dinov3_backbone = AutoModel.from_pretrained(pretrained_model_name).to(device)

# split 
train_data_path = os.path.join("..", "data", "combined_split", "train")
val_data_path = os.path.join("..", "data", "combined_split", "val")
test_data_path = os.path.join("..", "data", "combined_split", "test")

# dataset wrapper
class dinov3dataset():
    def __init__(self, root, processor):
        self.ds = datasets.ImageFolder(root)
        self.processor = processor
    
    def __len__(self):
        return len(self.ds)
    
    def __getitem__(self, idx):
        img, label = self.ds[idx]
        inputs = self.processor(images=img, return_tensors="pt")
        return {"pixel_values": inputs["pixel_values"].squeeze(0), "labels": torch.tensor(label)}
    
    def getsubset(self, frac):
        num_samples = int(len(self.ds) * frac)
        generator = torch.Generator().manual_seed(10)
        subset, _ = random_split(self, [num_samples, len(self.ds) - num_samples], generator)
        return subset
    
train_ds = dinov3dataset(train_data_path, dinov3_processor).getsubset(0.05)
val_ds = dinov3dataset(val_data_path, dinov3_processor)
test_ds = dinov3dataset(test_data_path, dinov3_processor)

num_classes = len(os.listdir(train_data_path))

# custom -> wrap backbone and head
class DINOv3Classifier(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(backbone.config.hidden_size, num_classes)

    def forward(self, pixel_values, labels=None):
        out = self.backbone(pixel_values=pixel_values)
        logits = self.classifier(out.last_hidden_state[:, 0])
        loss = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}
    
dinov3 = DINOv3Classifier(dinov3_backbone, num_classes)

# train
lr = 1e-5
num_epochs = 10
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=num_epochs,
    learning_rate=lr,
    save_steps=500,
    save_total_limit=2,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    f1 = f1_score(labels, preds, average="macro")
    acc = (preds == labels).mean()
    return {"f1": f1, "acc": acc}

# print per epoch while training
class EpochMetricsCallback(TrainerCallback):
    def __init__(self):
        self.metrics = {}

    def on_evaluate(self, args, state, control, metrics, **kwargs):
        self.metrics.update(metrics)

        if "eval_train_f1" in self.metrics and "eval_val_f1" in self.metrics:
            epoch = int(state.epoch)
            print(f"Epoch {epoch}/{num_epochs}, Training Loss: {self.metrics.get('eval_train_loss', 0):.4f}, Training Accuracy: {self.metrics.get('eval_train_acc', 0):.4f}, Training F1 Score: {self.metrics.get('eval_train_f1', 0):.4f}, Validation Loss: {self.metrics.get('eval_val_loss', 0):.4f}, Validation Accuracy: {self.metrics.get('eval_val_acc', 0):.4f}, Validation F1 Score: {self.metrics.get('eval_val_f1', 0):.4f}")
            run.log({"train loss": self.metrics.get('eval_train_loss', 0),"train acc": self.metrics.get('eval_train_acc', 0), "train f1 score": self.metrics.get('eval_train_f1', 0), "val loss": self.metrics.get('eval_val_loss', 0), "val acc": self.metrics.get('eval_val_acc', 0), "val f1 score": self.metrics.get('eval_val_f1', 0)})
            self.metrics = {}

dinov3_trainer = Trainer(
    model=dinov3,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset={"train": train_ds, "val": val_ds},
    compute_metrics=compute_metrics,
    callbacks=[EpochMetricsCallback()],
)

# evaluate on test set
def dinov3_evaluate(trainer, dataset):
    results = trainer.predict(dataset)
    preds = results.predictions.argmax(axis=-1)
    labels = results.label_ids
    f1 = f1_score(labels, preds, average="macro")
    acc = (preds == labels).mean()

    print(f"Final Average Accuracy: {acc:.4f}, Final F1 Score: {f1:.4f}")
    run.log({"final acc": acc, "final f1": f1})

import wandb
run = wandb.init(
    entity="rchan192-university-of-california-riverside",
    project="my-awesome-project",
    config={
        "learning_rate": lr,
        "architecture": "DINOv3",
        "dataset": "CitrusUAT + Orange Leaves for HLB",
        "epochs": num_epochs,
    },
)

dinov3_trainer.train()
dinov3_evaluate(dinov3_trainer, test_ds)

run.finish()

## AUTOMATED RUNS

In [6]:
import random as rand
import pandas as pd

# models
# resnet
# vgg-19
# clip
dinov2_model_name = "facebook/dinov2-base"
# dinov3

# epoch values per percentage
# resnet
# vgg-19
# clip
dinov2_epochs = [20, 6, 15, 15, 12, 9, 6, 10, 6, 4, 4] # will be fixed
# dinov3

# for each percentage
rng = rand.Random()
results = []
fracs = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
for n in range(len(fracs)):
    frac = fracs[n]

    print(f"Current: {frac * 100}%")

    # resnet
    # vgg-19
    # clip
    dinov2_res = []
    # dinov3

    # 10 runs each
    for i in range(10):
        # same random seed across models
        seed = rng.randint(0, 2**32 - 1)
        generator = torch.Generator().manual_seed(seed)
        
        # resnet

        # vgg-19

        # clip

        # dinov2
        dinov2_processor = AutoImageProcessor.from_pretrained(dinov2_model_name)
        dinov2_backbone = AutoModel.from_pretrained(dinov2_model_name).to(device)
        dinov2 = DINOv2Classifier(dinov2_backbone, num_classes)

        train_ds = dinov2dataset(train_data_path, dinov2_processor).getsubset(frac, generator)
        val_ds = dinov2dataset(val_data_path, dinov2_processor)
        test_ds = dinov2dataset(test_data_path, dinov2_processor)

        lr = 1e-5
        num_epochs = dinov2_epochs[n]
        training_args = TrainingArguments(
            output_dir="./results",
            eval_strategy="epoch",
            logging_strategy="epoch",
            per_device_train_batch_size=32,
            per_device_eval_batch_size=64,
            num_train_epochs=num_epochs,
            learning_rate=lr,
            save_steps=500,
            save_total_limit=2,
        )

        dinov2_trainer = Trainer(
            model=dinov2,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset={"train": train_ds, "val": val_ds},
            compute_metrics=compute_metrics,
            callbacks=[EpochMetricsCallback()],
        )

        dinov2_trainer.train()
        dinov2_res.append(dinov2_evaluate(dinov2_trainer, test_ds))

        del dinov2_processor, dinov2_backbone, dinov2

        # dinov3
    # combine runs
    # resnet
    # vgg-19
    # clip
    results.append({"Run": f"DINOv2 {frac * 100}%", "F1 Score": dinov2_res})
    # dinov3

df = pd.DataFrame(results)
df.to_csv('percentagetrials.csv', index=False)

Current: 5.0%
1734189912


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Train Loss,Train F1,Train Acc,Val Loss,Val F1,Val Acc
1,1.207351,No log,0.000548,1.000000,1.000000,2.367796,0.378378,0.608696
2,0.000548,No log,0.000005,1.000000,1.000000,4.032034,0.378378,0.608696
3,0.000005,No log,0.000001,1.000000,1.000000,4.882174,0.378378,0.608696
4,0.000001,No log,0.000000,1.000000,1.000000,5.368645,0.378378,0.608696
5,0.000000,No log,0.000000,1.000000,1.000000,5.677389,0.378378,0.608696
6,0.000000,No log,0.000000,1.000000,1.000000,5.907137,0.378378,0.608696
7,0.000000,No log,0.000000,1.000000,1.000000,6.072297,0.378378,0.608696
8,0.000000,No log,0.000000,1.000000,1.000000,6.193528,0.378378,0.608696
9,0.000000,No log,0.000000,1.000000,1.000000,6.284674,0.378378,0.608696
10,0.000000,No log,0.000000,1.000000,1.000000,6.355357,0.378378,0.608696


/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' ar

...

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Train Loss,Train F1,Train Acc,Val Loss,Val F1,Val Acc
1,0.407285,No log,0.024564,1.000000,1.000000,1.588037,0.378378,0.608696
2,0.024564,No log,0.004214,1.000000,1.000000,1.038594,0.581818,0.695652
3,0.004214,No log,0.000495,1.000000,1.000000,1.501522,0.581818,0.695652
4,0.000495,No log,0.000130,1.000000,1.000000,1.888263,0.581818,0.695652
5,0.000130,No log,0.000066,1.000000,1.000000,2.144226,0.581818,0.695652
6,0.000066,No log,0.000045,1.000000,1.000000,2.302454,0.581818,0.695652
7,0.000045,No log,0.000037,1.000000,1.000000,2.390632,0.581818,0.695652
8,0.000037,No log,0.000034,1.000000,1.000000,2.428543,0.581818,0.695652


/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' ar

Final Average Accuracy: 0.8276, Final F1 Score: 0.6712
